<a href="https://colab.research.google.com/github/jenyabydaev-web/homeworkk/blob/homework/4%EC%9B%94%20%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import gradio as gr


categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']
newsgroups = fetch_20newsgroups(subset='train', categories=categories,
                                remove=('headers', 'footers', 'quotes'))

data, labels = [], []
for i, category in enumerate(categories):

    indices = np.where(newsgroups.target == i)[0][:20]
    for idx in indices:
        data.append(newsgroups.data[idx])
        labels.append(newsgroups.target_names[i])


vectorizer = CountVectorizer(stop_words='english')
count_matrix = vectorizer.fit_transform(data)


def classify_text(input_text):

    input_vec = vectorizer.transform([input_text])

    sim = cosine_similarity(input_vec, count_matrix)


    best_idx = np.argmax(sim)
    return labels[best_idx], sim[0][best_idx]


def predict(text):
    if not text.strip():
        return "텍스트를 입력해주세요."

    cat, score = classify_text(text)

    if score == 0:
        return f"일치하는 단어가 없습니다. (유사도: {score:.4f})"
    return f"예측 카테고리: {cat} (유사도: {score:.4f})"


demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=2, placeholder="영문장이나 단어를 입력하세요..."),
    outputs="text",
    title="NLP 뉴스 그룹 분류 서비스",
    description="입력한 문장이 그래픽, 우주, 종교 중 어떤 주제에 가까운지 분석합니다."
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bbe5f75230cc974c2a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Q1. 유사도가 0.0000이 나오는 이유 분석 특정 문장(예: "Exploring the mars with a robotic
rover.")을 입력했을 때 유사도가 0이 나오거나 엉뚱한 카테고리가 매칭되는 이유를
CountVectorizer의 동작 원리와 관련지어 설명하세요.


원인: CountVectorizer는 학습 데이터(작성하신 60개의 샘플)에 포함된 단어들만을 바탕으로 단어 사전(vocabulary)을 구축합니다.

OOV(Out-of-Vocabulary) 문제: 입력한 문장에 학습 데이터에서 나타나지 않은 단어들만 포함되어 있다면, 해당 문장의 벡터는 모두 0으로 구성된 '희소 벡터(zero vector)'가 됩니다.

결과: 모든 값이 0인 벡터와 다른 벡터 사이의 코사인 유사도는 항상 0이 됩니다. 만약 의미가 없는 단어 하나만 우연히 일치하게 된다면, 시스템이 전혀 엉뚱한 카테고리를 선택하는 오류가 발생할 수 있습니다.

Q2. 성능 개선 실험 유사도 점수를 높이고 분류 정확도를 올리기 위해 다음 중 하나를 시도하고
결과를 비교하세요.

In [2]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import gradio as gr


categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']

newsgroups = fetch_20newsgroups(subset='train', categories=categories,
                                remove=('headers', 'footers', 'quotes'))

data, labels = [], []
for i, category in enumerate(categories):

    indices = np.where(newsgroups.target == i)[0][:100]
    for idx in indices:
        data.append(newsgroups.data[idx])
        labels.append(newsgroups.target_names[i])


vectorizer = CountVectorizer(stop_words='english')
count_matrix = vectorizer.fit_transform(data)


def classify_text(input_text):
    input_vec = vectorizer.transform([input_text])
    sim = cosine_similarity(input_vec, count_matrix)
    best_idx = np.argmax(sim)
    return labels[best_idx], sim[0][best_idx]


def predict(text):
    if not text.strip():
        return "텍스트를 입력해주세요."

    cat, score = classify_text(text)

    if score == 0:
        return f"일치하는 단어가 없습니다. (유사도: {score:.4f})"
    return f"예측 카테고리: {cat} (유사도: {score:.4f})"

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=2, placeholder="영문장이나 단어를 입력하세요..."),
    outputs="text",
    title="NLP 뉴스 그룹 분류 서비스",
    description="입력한 문장이 그래픽, 우주, 종교 중 어떤 주제에 가까운지 분석합니다."
)


demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://11d152f449bd9cb1e0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
